# UKB longevity-associated SNP manifest — first contact

Offline sanity checks on `analysis/ukb_snp_manifest_v0.1.csv` (from `scripts/ukb_la_snp_lookup.py`). At full build this table is on the order of ~70 longevity-associated SNPs across ~41 genes. Exploratory only: confirm the manifest looks usable before spending UK Biobank extraction credits.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

_here = Path.cwd()
_candidates = (
    _here / "analysis" / "ukb_snp_manifest_v0.1.csv",
    _here.parent / "analysis" / "ukb_snp_manifest_v0.1.csv",
    _here.parent.parent / "analysis" / "ukb_snp_manifest_v0.1.csv",
)
MANIFEST_PATH = next((p.resolve() for p in _candidates if p.exists()), _candidates[0])
if not MANIFEST_PATH.exists():
    raise FileNotFoundError("Generate the CSV with scripts/ukb_la_snp_lookup.py (repo root as cwd).")

manifest = pd.read_csv(MANIFEST_PATH)
print(MANIFEST_PATH)
print(manifest.shape)
manifest.head()


## 2. Coverage check

In [ ]:
chrom_series = manifest["Chromosome"].fillna("").astype(str).str.strip()
pos = manifest["Position_GRCh38"]
resolved = chrom_series.ne("") & pos.notna()

n_total = len(manifest)
n_resolved = int(resolved.sum())
n_failed = n_total - n_resolved
print(f"GRCh38 coordinates resolved: {n_resolved} / {n_total}")
print(f"Missing or failed: {n_failed}")
failed = manifest.loc[~resolved, ["Gene", "SNP_rsID", "Chromosome", "Position_GRCh38"]]
failed


## 3. Chromosome distribution

In [ ]:
def _chr_natural_key(ch: str) -> tuple[int, str]:
    u = str(ch).strip().upper()
    if u == "X":
        return (23, u)
    if u == "Y":
        return (24, u)
    if u in {"MT", "M"}:
        return (25, u)
    try:
        return (int(u), u)
    except ValueError:
        return (99, u)


In [ ]:
labels = sorted(
    manifest.loc[resolved, "Chromosome"].astype(str).str.strip().unique(),
    key=_chr_natural_key,
)
chrom_counts = (
    manifest.loc[resolved, "Chromosome"]
    .astype(str)
    .str.strip()
    .value_counts()
    .reindex(labels)
    .fillna(0)
    .astype(int)
)
chrom_counts


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(x=chrom_counts.index, y=chrom_counts.values, ax=ax, color="steelblue")
ax.set_title("SNPs per chromosome (resolved loci only)")
ax.set_xlabel("Chromosome")
ax.set_ylabel("SNP count")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()


Chromosome 6 often carries several longevity- and immune-related signals (including **HLA-DQB1** in the MHC); a modest excess of SNPs on chr6 is not unexpected in this manifest, but worth a second glance in the counts above.

## 4. Gene-level summary

In [ ]:
def _gene_chrom_rollup(s: pd.Series) -> str:
    u = sorted({str(x).strip() for x in s.dropna() if str(x).strip()})
    if not u:
        return "—"
    return u[0] if len(u) == 1 else ", ".join(u)


gene_summary = (
    manifest.groupby("Gene", dropna=False)
    .agg(n_SNPs=("SNP_rsID", "count"), Chromosome=("Chromosome", _gene_chrom_rollup))
    .reset_index()
    .sort_values("Gene", key=lambda c: c.str.lower())
)
gene_summary


## 5. UKB readiness (genomic span per chromosome)

In [ ]:
ok = manifest.loc[resolved].copy()
ok["Chromosome"] = ok["Chromosome"].astype(str).str.strip()
ok["Position_GRCh38"] = pd.to_numeric(ok["Position_GRCh38"], errors="coerce")
ranges = (
    ok.groupby("Chromosome", sort=False)["Position_GRCh38"]
    .agg(pos_min_GRCh38="min", pos_max_GRCh38="max")
    .reset_index()
    .sort_values("Chromosome", key=lambda s: s.map(lambda c: _chr_natural_key(str(c))[0]))
)
ranges


## 6. Brief summary

In [ ]:
from IPython.display import Markdown

body = f"""
### First-contact snapshot

| Metric | Value |
| --- | ---: |
| Total manifest rows | {n_total} |
| Resolved to GRCh38 | {n_resolved} |
| Missing / failed | {n_failed} |

**Suggested next step:** If failures remain, re-run the lookup script or inspect those rsIDs in Ensembl. If coverage is complete, map these GRCh38 intervals to your chosen UKB genotype product (note GRCh37 vs GRCh38) and plan chromosome-batched extraction using `UKB_Expected_Chunk` as a planning label only.
"""
display(Markdown(body))
